# 记忆基准测试（LoCoMo & LongMemEval）

> **测不了就改进不了。LoCoMo 和 LongMemEval 提供标准化的测试套件，准确告诉你记忆系统在哪里失败：单跳检索、多跳推理、时序排序，还是开放式生成。**


## 动机

想象你雇了一位新助理，入职一周后给他做一套测验。第一部分考他记不记得你点的咖啡（直接检索）。第二部分考他能不能把两次分开的谈话联系起来，替你安排日程（多跳推理）。第三部分考他知不知道你上周三从茶换成了咖啡（时序感知）。总分有用，但分项得分才告诉你具体该练什么。

**LoCoMo**（Long-Context Conversational Memory）和 **LongMemEval** 是标准化基准测试，对 AI 记忆系统起同样的作用。它们提供多会话对话，并配上标准答案（ground truth）问题。每个问题针对一种特定的记忆能力。用你的系统跑这些基准测试，会得到分类别得分，把「记忆质量」从一句模糊的口号变成一个可以跟踪、可以改进的具体数字。

每个智能体记忆系统都声称能「记住」对话。但它能回想起 50 条消息之前提到的偏好吗？它能连接两个不同会话里的事实吗？它知道两句矛盾的话哪句是后说的吗？没有基准测试，你只能靠猜。有了它，你才知道。

**在本 notebook 中，你将：**
1. 加载 LoCoMo 基准数据集并探索其结构。
2. 构建一个记忆系统适配器，用来摄取基准对话。
3. 运行预测流程，用检索到的上下文回答基准问题。
4. 用 BLEU、ROUGE-L、F1 和 LLM 评委给预测打分。
5. 分析分类别结果，并与无记忆基线对比。


## 关键概念

- **LoCoMo 基准**：一个包含 10 段多会话对话的数据集，约有 2,000 组问答对。每段对话模拟两个人跨最多 35 个会话聊天。问题测试五大类记忆能力。「基准测试」指用来衡量性能的标准化测试。
- **LongMemEval**：一个互补的基准，含 500 个问答实例。每个实例是一段用户与助手的聊天历史，横跨 40 到 500 个会话。它测试五项核心记忆能力，包括拒答（abstention，即知道某些信息从未被提及）。「拒答」指系统正确地拒绝回答，而不是瞎猜。
- **单跳问题（single-hop）**：只需从一个会话检索一条事实的问题。例如：「Caroline 最喜欢什么颜色？」答案出现在单独一轮对话里。
- **多跳问题（multi-hop）**：需要连接两个或更多会话事实的问题。例如：「Caroline 提到的同事里，谁也参加了那个会议？」你至少需要两段不同对话里的信息。
- **时序问题（temporal）**：关于事情何时发生、或随时间有何变化的问题。例如：「用户转行之前做什么工作？」系统必须理解先后顺序和时间戳。
- **开放式问题（open-ended）**：需要从散落在各会话的事实中综合出答案的问题。没有唯一正确的说法，答案必须连贯且完整。
- **对抗性问题（adversarial）**：故意误导的问题。它可能提到从未发生过的事件，或与对话内容矛盾。好的记忆系统应该识破陷阱。
- **BLEU 分数**：一个指标（0 到 1 之间的数字），衡量预测答案里有多少词序列与参考答案重合。越高代表重合越多。「n-gram」指 N 个连续单词组成的序列。
- **ROUGE-L 分数**：衡量预测与参考之间最长公共子序列的指标。它侧重查全率（参考答案被覆盖了多少）。「子序列」指按相同顺序出现、但不一定相邻的单词。
- **F1 分数**：平衡查准率（预测的词有多少是对的）和查全率（参考的词有多少被覆盖）的指标。它取两者的调和平均。
- **LLM 评委**：用一个能力强的模型（如 GPT-4 或 Claude）来判断自由格式的答案是否正确。它能处理措辞不同、但含义一致的情况。


## 架构

<p align="center">
  <img src="../../images/diagrams/29_memory_benchmarks_LoCoMo.svg" alt="架构图" width="720"/>
</p>

这张图展示完整的评估流程。基准数据（对话、问题和标准答案）流入被测的记忆系统。系统先摄取对话，再用检索机制回答每个问题。预测结果进入评分流程，计算出 BLEU、ROUGE、F1 和 LLM 评委分数。结果再拆解成分类别得分、弱点分析和基线对比。

关键点在于：你用和生产环境完全一样的方式测试系统，只是这次正确答案事先已知。这样，你每迭代一次记忆架构，都能度量出改进了多少。


## 准备工作

安装所需的包。我们用 `datasets` 从 HuggingFace 加载 LoCoMo，用 `rouge-score` 和 `nltk` 计算文本指标，用 `openai` 调用 LLM 并做 LLM 评委打分。


In [ ]:
%pip install -q python-dotenv datasets rouge-score nltk deepagents langchain-openai

导入所有依赖并配置 API 访问。你需要设置 `LLM_API_KEY` 环境变量。


In [ ]:
import os
import json
import re
import collections

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 实现

### 第 1 步：加载并探索 LoCoMo 数据集

LoCoMo 托管在 HuggingFace 上。每个样本包含两位说话人的多会话对话，外加按类别组织的问题答对。我们先加载一个样本，看看它的结构。


In [ ]:
# 从 HuggingFace 加载 LoCoMo 数据集
locomo_dataset = load_dataset("snap-stanford/LoCoMo", split="train")

print(f"Number of conversation samples: {len(locomo_dataset)}")
print(f"Fields per sample: {list(locomo_dataset[0].keys())}")

我们来看一个对话样本。每个样本包含若干会话（编号的对话轮次）、一组问答对，以及事件摘要等辅助元数据。


In [ ]:
# 取第一个对话样本
sample = locomo_dataset[0]

# 解析对话 JSON
conversation = json.loads(sample["conversation"])

print(f"Speaker A: {conversation['speaker_a']}")
print(f"Speaker B: {conversation['speaker_b']}")

# 通过会话键统计会话数量
session_keys = sorted([k for k in conversation if k.startswith("session_") and not k.endswith("_date_time")])
print(f"Number of sessions: {len(session_keys)}")

# 展示第一个会话的前几轮
first_session = conversation[session_keys[0]]
print(f"\nFirst session has {len(first_session)} turns:")
for turn in first_session[:3]:
    speaker = turn["speaker"]
    text = turn["text"][:100] + ("..." if len(turn["text"]) > 100 else "")
    print(f"  {speaker}: {text}")

接下来看问答对。每个 QA 有一个类别（1-5）、一个问题、一个标准答案，以及指向具体对话轮次的证据（evidence）标注。


In [ ]:
# 解析问答对
qa_pairs = json.loads(sample["qa"])

# 类别映射
CATEGORY_NAMES = {
    1: "single-hop",
    2: "temporal",
    3: "multi-hop",
    4: "open-ended",
    5: "adversarial"
}

# 按类别统计问题数
category_counts = collections.Counter(q["category"] for q in qa_pairs)
print("Question counts by category:")
for cat_id in sorted(category_counts):
    name = CATEGORY_NAMES.get(cat_id, f"unknown-{cat_id}")
    print(f"  Category {cat_id} ({name}): {category_counts[cat_id]}")

print(f"\nTotal QA pairs in this sample: {len(qa_pairs)}")

# 展示每个类别的问题示例
print("\n--- Example questions ---")
shown = set()
for q in qa_pairs:
    cat = q["category"]
    if cat not in shown:
        shown.add(cat)
        answer_text = str(q["answer"])[:80]
        print(f"\n[{CATEGORY_NAMES.get(cat, cat)}]")
        print(f"  Q: {q['question']}")
        print(f"  A: {answer_text}")
        print(f"  Evidence: {q.get('evidence', 'N/A')}")

### 第 2 步：构建对话上下文

在回答问题之前，我们要把结构化对话转成纯文本。这模拟的是记忆系统会存储和检索的内容。我们将构建完整的对话文本，带上会话标记和时间戳。


In [ ]:
def build_conversation_text(conversation_json):
    """把 LoCoMo 对话字典转换成带会话标记的纯文本。"""
    conversation = json.loads(conversation_json) if isinstance(conversation_json, str) else conversation_json

    session_keys = sorted(
        [k for k in conversation if k.startswith("session_") and not k.endswith("_date_time")],
        key=lambda k: int(k.split("_")[1])
    )

    parts = []
    for session_key in session_keys:
        session_num = session_key.split("_")[1]
        date_key = f"{session_key}_date_time"
        timestamp = conversation.get(date_key, "unknown date")

        parts.append(f"\n=== Session {session_num} ({timestamp}) ===")

        for turn in conversation[session_key]:
            parts.append(f"{turn['speaker']}: {turn['text']}")

    return "\n".join(parts)


# Build context for our sample
full_context = build_conversation_text(sample["conversation"])
print(f"Full conversation length: {len(full_context)} characters")
print(f"Approximate tokens: ~{len(full_context) // 4}")
print(f"\nFirst 500 characters:")
print(full_context[:500])

### 第 3 步：构建预测流程

对每个问题，我们把对话上下文连同问题一起发给 LLM。模型根据它在上下文里找到的内容生成答案。这就是「全上下文基线」做法：把整段对话作为上下文传入。

在生产环境的记忆系统里，你会把这一步换成自己的检索机制。基准测试保持不变，变的只是上下文选择这一步。


In [ ]:
def generate_prediction(question, context, model=MODEL):
    """给定对话上下文，为基准问题生成答案。"""
    system_prompt = (
        "You are answering questions about a conversation history. "
        "Use only the information in the provided conversation. "
        "If the answer is not in the conversation, say 'I don't know'. "
        "Keep your answer brief and direct (1-2 sentences max)."
    )

    user_prompt = (
        f"Conversation history:\n{context}\n\n"
        f"Question: {question}\n\n"
        f"Answer based only on the conversation above:"
    )

    response = llm.invoke([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ])

    return to_text(response.content).strip()


# Test with one question
test_q = qa_pairs[0]
test_answer = generate_prediction(test_q["question"], full_context)
print(f"Question:   {test_q['question']}")
print(f"Predicted:  {test_answer}")
print(f"Reference:  {test_q['answer']}")

现在对一部分问题运行预测。我们限制为 30 个问题（按类别抽样），以控制 API 开销。完整评估时，你会跑所有问题。


In [ ]:
import random

random.seed(42)

# 最多抽取 30 个问题，按类别 1-4 分层抽样
# （类别 5 是对抗性问题，通常不参与打分）
scorable_qs = [q for q in qa_pairs if q["category"] in [1, 2, 3, 4]]
sample_size = min(30, len(scorable_qs))
sampled_qs = random.sample(scorable_qs, sample_size)

print(f"Running predictions for {sample_size} questions...")
category_sample_counts = collections.Counter(q["category"] for q in sampled_qs)
for cat_id in sorted(category_sample_counts):
    print(f"  {CATEGORY_NAMES[cat_id]}: {category_sample_counts[cat_id]}")

predictions = []
for i, q in enumerate(sampled_qs):
    pred = generate_prediction(q["question"], full_context)
    predictions.append({
        "question": q["question"],
        "prediction": pred,
        "reference": str(q["answer"]),
        "category": q["category"],
        "evidence": q.get("evidence", []),
    })
    if (i + 1) % 10 == 0:
        print(f"  Completed {i + 1}/{sample_size}")

print(f"\nAll {sample_size} predictions generated.")

### 第 4 步：构建评分流程

我们为每个预测计算三个自动指标：

1. **BLEU-1**：预测与参考之间的 unigram（单词级）重合度。
2. **ROUGE-L**：最长公共子序列的重合度。
3. **token F1**：词级别的查准率与查全率。

每个指标捕捉答案质量的不同侧面。BLEU 侧重查准率（预测的词对不对？）。ROUGE 侧重查全率（参考的词覆盖了没有？）。F1 平衡两者。


In [ ]:
def compute_token_f1(prediction, reference):
    """计算预测与参考字符串之间的 token 级 F1。"""
    pred_tokens = prediction.lower().split()
    ref_tokens = reference.lower().split()

    if not pred_tokens or not ref_tokens:
        return 0.0

    common = collections.Counter(pred_tokens) & collections.Counter(ref_tokens)
    num_common = sum(common.values())

    if num_common == 0:
        return 0.0

    precision = num_common / len(pred_tokens)
    recall = num_common / len(ref_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return f1


def compute_bleu1(prediction, reference):
    """计算 BLEU-1（unigram 查准率）分数。"""
    pred_tokens = prediction.lower().split()
    ref_tokens = reference.lower().split()

    if not pred_tokens or not ref_tokens:
        return 0.0

    ref_counts = collections.Counter(ref_tokens)
    clipped = 0
    for token in pred_tokens:
        if ref_counts[token] > 0:
            clipped += 1
            ref_counts[token] -= 1

    return clipped / len(pred_tokens) if pred_tokens else 0.0


# 初始化 ROUGE 评分器
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def compute_rouge_l(prediction, reference):
    """计算 ROUGE-L F 值。"""
    scores = scorer.score(reference, prediction)
    return scores["rougeL"].fmeasure


# Test metrics on one example
test_pred = predictions[0]
print(f"Question:  {test_pred['question']}")
print(f"Predicted: {test_pred['prediction']}")
print(f"Reference: {test_pred['reference']}")
print(f"\nToken F1:  {compute_token_f1(test_pred['prediction'], test_pred['reference']):.3f}")
print(f"BLEU-1:    {compute_bleu1(test_pred['prediction'], test_pred['reference']):.3f}")
print(f"ROUGE-L:   {compute_rouge_l(test_pred['prediction'], test_pred['reference']):.3f}")

### 第 5 步：LLM 评委打分

自动文本指标会漏掉措辞不同、但意思正确的情况。比如「New York City」和「NYC」的词重合度很低，含义却完全一样。LLM 评委可以评估语义正确性。

我们让一个能力强的模型来判断：「给定参考答案，这个预测正确吗？」评委返回一个二值分数（正确为 1，错误为 0）。


In [ ]:
def llm_judge_score(question, prediction, reference, model=JUDGE_MODEL):
    """用 LLM 判断预测在语义上是否正确。"""
    prompt = (
        "You are evaluating an answer to a question about a conversation.\n\n"
        f"Question: {question}\n"
        f"Reference answer: {reference}\n"
        f"Predicted answer: {prediction}\n\n"
        "Is the predicted answer correct? It doesn't need to match word-for-word, "
        "but it must convey the same factual information as the reference.\n"
        "Reply with exactly 'CORRECT' or 'INCORRECT'."
    )

    response = llm.invoke([{"role": "user", "content": prompt}])

    reply = to_text(response.content).strip().upper()
    return 1 if "CORRECT" in reply else 0


# Test the judge on a few examples
for p in predictions[:3]:
    judge = llm_judge_score(p["question"], p["prediction"], p["reference"])
    status = "CORRECT" if judge == 1 else "INCORRECT"
    print(f"Q: {p['question'][:60]}...")
    print(f"  Pred: {p['prediction'][:60]}...")
    print(f"  Ref:  {p['reference'][:60]}...")
    print(f"  Judge: {status}\n")

### 第 6 步：按类别汇总得分

现在我们为每个预测计算所有指标，并按问题类别分组汇总。基准测试的诊断能力在这里体现：分类别得分直接暴露具体弱点。


In [ ]:
def score_all_predictions(predictions):
    """给所有预测打分，返回分类别和总体指标。"""
    results_by_category = collections.defaultdict(list)

    for p in predictions:
        f1 = compute_token_f1(p["prediction"], p["reference"])
        bleu = compute_bleu1(p["prediction"], p["reference"])
        rouge = compute_rouge_l(p["prediction"], p["reference"])
        judge = llm_judge_score(p["question"], p["prediction"], p["reference"])

        results_by_category[p["category"]].append({
            "f1": f1,
            "bleu1": bleu,
            "rougeL": rouge,
            "judge": judge,
        })

    # 按类别汇总
    summary = {}
    for cat_id, results in sorted(results_by_category.items()):
        n = len(results)
        summary[cat_id] = {
            "name": CATEGORY_NAMES.get(cat_id, f"cat-{cat_id}"),
            "count": n,
            "f1": sum(r["f1"] for r in results) / n,
            "bleu1": sum(r["bleu1"] for r in results) / n,
            "rougeL": sum(r["rougeL"] for r in results) / n,
            "judge_accuracy": sum(r["judge"] for r in results) / n,
        }

    # 总体（按数量加权）
    total = sum(s["count"] for s in summary.values())
    overall = {
        "name": "overall",
        "count": total,
        "f1": sum(s["f1"] * s["count"] for s in summary.values()) / total,
        "bleu1": sum(s["bleu1"] * s["count"] for s in summary.values()) / total,
        "rougeL": sum(s["rougeL"] * s["count"] for s in summary.values()) / total,
        "judge_accuracy": sum(s["judge_accuracy"] * s["count"] for s in summary.values()) / total,
    }

    return summary, overall


print("Scoring all predictions (this calls the LLM judge for each)...")
category_scores, overall_scores = score_all_predictions(predictions)

# Display results
print(f"\n{'Category':<14} {'Count':>5} {'F1':>7} {'BLEU-1':>7} {'ROUGE-L':>8} {'Judge':>7}")
print("-" * 52)
for cat_id, scores in sorted(category_scores.items()):
    print(f"{scores['name']:<14} {scores['count']:>5} "
          f"{scores['f1']:>7.3f} {scores['bleu1']:>7.3f} "
          f"{scores['rougeL']:>8.3f} {scores['judge_accuracy']:>7.3f}")
print("-" * 52)
print(f"{'overall':<14} {overall_scores['count']:>5} "
      f"{overall_scores['f1']:>7.3f} {overall_scores['bleu1']:>7.3f} "
      f"{overall_scores['rougeL']:>8.3f} {overall_scores['judge_accuracy']:>7.3f}")

### 第 7 步：与无记忆基线对比

基准测试的分数只有在对比中才有意义。我们来建一个「无记忆」基线：LLM 在没有任何对话上下文的情况下回答每个问题。这能看出记忆系统带来了多少价值。

如果你的记忆系统在某个类别上得分和无记忆基线接近，说明你的检索对这类问题没有帮助。


In [ ]:
def generate_no_memory_prediction(question, model=MODEL):
    """在没有对话上下文的情况下回答问题（基线）。"""
    response = llm.invoke([
            {"role": "system", "content": "Answer the question briefly. If you don't know, say 'I don't know'."},
            {"role": "user", "content": question},
        ])
    return to_text(response.content).strip()


# 在相同问题上运行无记忆基线
print("Running no-memory baseline...")
baseline_predictions = []
for i, q in enumerate(sampled_qs):
    pred = generate_no_memory_prediction(q["question"])
    baseline_predictions.append({
        "question": q["question"],
        "prediction": pred,
        "reference": str(q["answer"]),
        "category": q["category"],
    })
    if (i + 1) % 10 == 0:
        print(f"  Completed {i + 1}/{sample_size}")

print("Scoring baseline...")
baseline_cat_scores, baseline_overall = score_all_predictions(baseline_predictions)

# 并排对比
print(f"\n{'Category':<14} {'Memory Judge':>13} {'No-Memory Judge':>16} {'Delta':>7}")
print("-" * 54)
for cat_id in sorted(category_scores):
    mem_score = category_scores[cat_id]["judge_accuracy"]
    base_score = baseline_cat_scores.get(cat_id, {}).get("judge_accuracy", 0.0)
    delta = mem_score - base_score
    name = CATEGORY_NAMES.get(cat_id, f"cat-{cat_id}")
    print(f"{name:<14} {mem_score:>13.3f} {base_score:>16.3f} {delta:>+7.3f}")
print("-" * 54)
delta_overall = overall_scores["judge_accuracy"] - baseline_overall["judge_accuracy"]
print(f"{'overall':<14} {overall_scores['judge_accuracy']:>13.3f} "
      f"{baseline_overall['judge_accuracy']:>16.3f} {delta_overall:>+7.3f}")

## 运行示例

我们走一遍具体的端到端例子。从每个类别（1-4）各挑一个问题，展示检索到的上下文片段、预测、参考答案，以及所有指标得分。这样评估过程就看得见摸得着了。


In [ ]:
# 每个类别挑一个问题，做详细演示
print("=" * 70)
print("DETAILED EXAMPLE: One question per category")
print("=" * 70)

shown_cats = set()
for p in predictions:
    cat = p["category"]
    if cat in shown_cats:
        continue
    shown_cats.add(cat)

    f1 = compute_token_f1(p["prediction"], p["reference"])
    bleu = compute_bleu1(p["prediction"], p["reference"])
    rouge = compute_rouge_l(p["prediction"], p["reference"])
    judge = llm_judge_score(p["question"], p["prediction"], p["reference"])

    print(f"\n--- {CATEGORY_NAMES[cat].upper()} (category {cat}) ---")
    print(f"Question:  {p['question']}")
    print(f"Reference: {p['reference']}")
    print(f"Predicted: {p['prediction']}")
    print(f"Evidence:  {p.get('evidence', 'N/A')}")
    print(f"Scores:    F1={f1:.3f}  BLEU-1={bleu:.3f}  ROUGE-L={rouge:.3f}  Judge={'CORRECT' if judge else 'INCORRECT'}")

    if len(shown_cats) >= 4:
        break

## LongMemEval：一个互补的基准

LongMemEval 用了另一种思路。它不用朋友之间的对话，而是用用户与助手的聊天历史。每个实例有 40 到 500 个会话，测试七种问题类型，归入五项记忆能力：

| 问题类型 | 记忆能力 | 测试内容 |
|---|---|---|
| single-session-user | 信息提取 | 回想单个会话中用户消息里的事实 |
| single-session-assistant | 信息提取 | 回想单个会话中助手消息里的事实 |
| single-session-preference | 信息提取 | 从单个会话推断隐含偏好 |
| multi-session | 多会话推理 | 连接多个会话的事实 |
| knowledge-update | 知识更新 | 识别随时间变化的信息 |
| temporal-reasoning | 时序推理 | 基于时间戳做时间推断 |
| （拒答变体） | 拒答（abstention） | 信息从未提及时正确拒绝回答 |

评估方法和前面一样：加载数据、摄取进记忆系统、生成预测、然后打分。LongMemEval 用 LLM 评委（GPT-4o）作为主要指标。

加载和查看它的方法如下：


In [ ]:
# 从 HuggingFace 加载 LongMemEval
# "oracle" 切分只含证据会话（最小，适合测试）
longmemeval = load_dataset("xiaowu0162/longmemeval-cleaned", split="longmemeval_oracle")

print(f"Number of instances: {len(longmemeval)}")
print(f"Fields: {list(longmemeval[0].keys())}")

# 查看一个实例
instance = longmemeval[0]
print(f"\nQuestion type: {instance['question_type']}")
print(f"Question: {instance['question']}")
print(f"Answer: {instance['answer']}")
print(f"Question date: {instance['question_date']}")
print(f"Number of sessions: {len(instance['haystack_sessions'])}")
print(f"Evidence session IDs: {instance['answer_session_ids']}")

# 按问题类型统计
type_counts = collections.Counter(inst["question_type"] for inst in longmemeval)
print("\nQuestion type distribution:")
for qtype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {qtype}: {count}")

要在 LongMemEval 上评估你的记忆系统，遵循同样的模式：摄取会话、用你的检索回答每个问题，再用 LLM 评委打分。数据集在单条消息上有 `has_answer` 字段，你还可以用它度量检索查全率（你的系统有没有找对相关轮次）。


## 权衡

### 记忆基准测试什么时候好用

- **诊断弱点。**分类别得分精确指出哪种记忆能力需要改进。单一的总体分数会掩盖这个信息。
- **对比方案。**你可以在同一批问题上客观比较两种记忆架构，比如向量检索和图记忆。
- **跟踪进展。**每次改动记忆系统后跑一次基准测试。分数上升说明改动有效；分数下降说明你引入了回归（regression，让事情变糟的改动）。
- **标准化沟通。**「我们的系统在 LoCoMo 多跳问题上得 78%」比「我们的系统跨会话推理不错」清楚得多。

### 什么时候会失灵

- **分布不匹配。**LoCoMo 的对话是朋友间聊生活。如果你的系统处理技术支持工单，基准分数可能反映不了真实表现。对话风格和你的用例越接近，分数才越有参考价值。
- **完整评估的成本。**把 2,000 个 LoCoMo 问题全部跑一遍记忆系统加 LLM 评委，要花真金白银的 API 费用。用 GPT-4o-mini 跑一次完整评估，预算大约 5-15 美元。
- **指标的局限。**文本重合度指标（BLEU、ROUGE、F1）会惩罚换了措辞的正确答案。LLM 评委能补救，但也不完美。一定要人工抽查一部分被判为「错误」的预测。
- **静态快照。**这些基准测试的是一组固定的对话和问题，覆盖不了你的系统在生产环境会遇到的所有边界情况。把它们当下限，不要当上限。


## 延伸阅读

- [Maharana et al., "Evaluating Very Long-Term Conversational Memory of LLM Agents," 2024 (arXiv:2402.17753)](https://arxiv.org/abs/2402.17753) - LoCoMo 论文。定义了该基准、问题分类法，以及在多种记忆架构上的基线评估。
- [LoCoMo GitHub 仓库](https://github.com/snap-research/LoCoMo) - 基准数据、评估脚本和基线实现。
- [Wu et al., "LongMemEval: Benchmarking Chat Assistants on Long-Term Interactive Memory," 2024 (arXiv:2410.10813)](https://arxiv.org/abs/2410.10813) - LongMemEval 论文。涵盖五项核心记忆能力，以及对商业聊天助手的评估。
- [LongMemEval GitHub 仓库](https://github.com/xiaowu0162/LongMemEval) - 基准数据、评估代码和结果。
- [HuggingFace Evaluate 库](https://huggingface.co/docs/evaluate?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - BLEU、ROUGE、F1 等指标的标准化实现。


*[← 上一篇：[28 - 记忆评估](../28_memory_evaluation/) · 下一篇：[30 - 生产级记忆模式](../30_production_memory_patterns/) →]*


## 🧪 动手试一试

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：分类别得分拆解
运行 `score_all_predictions()`，按 LoCoMo 的 5 个问题类别分组统计结果：单跳、多跳、时序、开放式、对抗性。打印一张表，列出每个类别的 F1 和 ROUGE-L。找出你的系统表现最差的类别。

### 挑战 2：上下文窗口消融实验（ablation）
改变喂给 `generate_prediction()` 的检索上下文段数：分别试 1、3、5、10 段。对每种设置，计算平均 token F1 和 `llm_judge_score()`。画出段数与得分的曲线，找到收益递减的拐点。

### 挑战 3：双系统对比基准
在两个不同的记忆系统上跑同一套 LoCoMo 评估（比如关键词重合检索 vs. 基于嵌入的检索）。两者都用 `score_all_predictions()` 打分，并排比较分类别结果。这直接建立在 28 记忆评估的评估框架之上。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--29-memory-benchmarks-locomo--memory-benchmarks-locomo)